# BERT Mechanistic Interpretability Experiment

**Subject:** `dog` | **Analyzed Model:** `bert-base-uncased` | **Explainer:** Llama 3 8B (local via Ollama)

---

This notebook implements the following experiment proposal in 5 steps:

1. **Choose one subject** — `dog`
2. **Generate a small set of prompts** — 10 total, 50% containing `dog`, 50% structurally matched controls
3. **Run the prompts through the model** — BERT (`bert-base-uncased`), capturing FFN layer-8 neuron activations via MPS-accelerated forward hooks
4. **Use the explainer LLM** — Local Llama 3 8B (via Ollama) analyzes the top differentially active neurons to determine whether they are associated with predicting `dog`
5. **Report results with concrete evidence** — Real activation values, a ranked neuron table, and the full LLM explanation

Each subsequent section is labeled with the proposal step it implements.

## Setup

Loading libraries and configuring MPS-accelerated inference on Apple Silicon.

We load PyTorch, HuggingFace Transformers, and the Ollama Python client. Device detection
follows the priority: **MPS** (Apple Silicon GPU) → **CPU**.
BERT will run on the selected device; Ollama manages its own MPS acceleration internally.

In [2]:
import datetime
import torch
import ollama
from transformers import BertTokenizer, BertModel

# --- Constants ---
LAYER    = 8        # BERT layer to hook (0-indexed); layers 7-9 are semantic-rich
TOP_N    = 15       # number of top differentially active neurons to inspect
MODEL_ID = 'bert-base-uncased'

# --- Device detection: MPS → CUDA → CPU ---
if torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
print(f'Device : {device}')
print(f'PyTorch: {torch.__version__}')
print(f'Run at : {ts}')

Device : mps
PyTorch: 2.10.0
Run at : 2026-03-07 17:26:18


## Step 2 — Generate Prompts

Generating a small set of prompts with 50% containing the target object (`dog`) and 50% not containing it.

We define **10 prompts** total:
- **5 dog prompts** — sentences where `dog` is the subject or key noun
- **5 control prompts** — the same sentence frames with varied animate nouns (cat, rabbit, bird, horse, fox)

Using varied control nouns prevents any single non-dog concept from becoming the spurious
contrasting class. Structural parallelism between dog and control prompts ensures that
syntactic position and sentence register are held constant — only the concept differs.

In [3]:
dog_prompts = [
    'The dog ran across the yard and barked loudly.',
    'She fed her dog a treat after the long walk.',
    'A large dog sat beside the front door waiting.',
    'He trained his dog to sit and stay on command.',
    'The dog curled up on the couch next to its owner.',
]

control_prompts = [
    'The cat ran across the yard and meowed loudly.',
    'She fed her rabbit a treat after the long walk.',
    'A large bird sat beside the front door waiting.',
    'He trained his horse to sit and stay on command.',
    'The fox curled up on the couch next to its owner.',
]

print('Dog prompts:')
for i, p in enumerate(dog_prompts, 1):
    print(f'  {i}. {p}')
print()
print('Control prompts:')
for i, p in enumerate(control_prompts, 1):
    print(f'  {i}. {p}')

Dog prompts:
  1. The dog ran across the yard and barked loudly.
  2. She fed her dog a treat after the long walk.
  3. A large dog sat beside the front door waiting.
  4. He trained his dog to sit and stay on command.
  5. The dog curled up on the couch next to its owner.

Control prompts:
  1. The cat ran across the yard and meowed loudly.
  2. She fed her rabbit a treat after the long walk.
  3. A large bird sat beside the front door waiting.
  4. He trained his horse to sit and stay on command.
  5. The fox curled up on the couch next to its owner.


## Step 3 — Run Prompts Through the Model

Running the prompts through the model.

We load `bert-base-uncased` onto the selected device and register a **PyTorch forward hook**
on `model.encoder.layer[8].intermediate` — the `BertIntermediate` module.
This module applies a linear projection (768 → 3072) followed by GELU activation,
producing the post-GELU FFN neuron activations. Its output shape is `(batch=1, seq_len, 3072)`.

We **mean-pool over the sequence dimension** to get a single `(3072,)` vector per prompt,
representing the average firing rate of each FFN neuron for that input.

**Why layer 8?** Tenney et al. (2019) showed that BERT's semantic and entity-level
representations peak in layers 7–9, making this an ideal target for concept-level analysis.

In [5]:
tokenizer = BertTokenizer.from_pretrained(MODEL_ID)
model = BertModel.from_pretrained(MODEL_ID).to(device)
model.eval()
print(f'Loaded {MODEL_ID} on {device}')

# Register a forward hook on BertIntermediate at layer LAYER
# BertIntermediate: linear (768->3072) + GELU -> post-GELU neuron activations
activation_store = {}

def hook_fn(module, input, output):
    # output shape: (1, seq_len, 3072)
    # mean-pool over sequence tokens -> (3072,), then move to CPU
    activation_store['current'] = output.detach().mean(dim=1).squeeze(0).cpu()

handle = model.encoder.layer[LAYER].intermediate.register_forward_hook(hook_fn)

# Run all 10 prompts through BERT
dog_acts = []
control_acts = []

with torch.no_grad():
    for prompt in dog_prompts:
        inputs = tokenizer(prompt, return_tensors='pt').to(device)
        model(**inputs)
        dog_acts.append(activation_store['current'])
    for prompt in control_prompts:
        inputs = tokenizer(prompt, return_tensors='pt').to(device)
        model(**inputs)
        control_acts.append(activation_store['current'])

handle.remove()  # deregister hook

print(f'Activations captured: {len(dog_acts)} dog, {len(control_acts)} control')
print(f'Shape per prompt    : {dog_acts[0].shape}  (3072 FFN neurons, mean-pooled over tokens)')
print(f'Dog prompt 1 norm   : {dog_acts[0].norm().item():.4f}')
print(f'Ctrl prompt 1 norm  : {control_acts[0].norm().item():.4f}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded bert-base-uncased on mps
Activations captured: 5 dog, 5 control
Shape per prompt    : torch.Size([3072])  (3072 FFN neurons, mean-pooled over tokens)
Dog prompt 1 norm   : 4.8476
Ctrl prompt 1 norm  : 5.1256


## Step 4 (Part A) — Identify Differentially Active Neurons

Using the explainer LLM to determine whether it can identify neurons associated with predicting the target object. This cell performs the neuron ranking that will be fed to the explainer.*

We compute:
- `dog_mean` — mean activation per neuron across all 5 dog prompts, shape `(3072,)`
- `control_mean` — mean activation per neuron across all 5 control prompts, shape `(3072,)`
- `diff = dog_mean − control_mean` — signed difference per neuron
- Rank all 3072 neurons by `|diff|` descending; take the top 15

**Positive diff (DOG>)** → neuron fires more for dog prompts than controls.  
**Negative diff (DOG<)** → neuron is suppressed for dog prompts relative to controls.

We use absolute mean difference rather than a t-test: 5 samples per class is far too few
for reliable p-values. Absolute difference is more transparent at this scale.

In [6]:
dog_mean     = torch.stack(dog_acts).mean(dim=0)      # (3072,)
control_mean = torch.stack(control_acts).mean(dim=0)  # (3072,)
diff         = dog_mean - control_mean                 # (3072,)
top_indices  = diff.abs().argsort(descending=True)[:TOP_N]

print(f'Overall dog neuron mean  : {dog_mean.mean().item():.4f}  (std: {dog_mean.std().item():.4f})')
print(f'Overall ctrl neuron mean : {control_mean.mean().item():.4f}  (std: {control_mean.std().item():.4f})')
print(f'Max |diff|               : {diff.abs().max().item():.4f}  at neuron {diff.abs().argmax().item()}')
print()
print('Rank  Neuron    Dog Mean   Ctrl Mean        Diff  Dir')
print('-' * 56)
for rank, idx in enumerate(top_indices, 1):
    i = idx.item()
    d = diff[i].item()
    direction = 'DOG>' if d > 0 else 'DOG<'
    print(f'{rank:>4}  {i:>6}  {dog_mean[i].item():>10.4f}  {control_mean[i].item():>10.4f}  {d:>10.4f}  {direction}')

Overall dog neuron mean  : -0.0410  (std: 0.0577)
Overall ctrl neuron mean : -0.0420  (std: 0.0550)
Max |diff|               : 0.1916  at neuron 1490

Rank  Neuron    Dog Mean   Ctrl Mean        Diff  Dir
--------------------------------------------------------
   1    1490      0.3087      0.1171      0.1916  DOG>
   2    2004      0.2275      0.0423      0.1851  DOG>
   3     468      0.2797      0.4502     -0.1704  DOG<
   4    2658      0.4118      0.2427      0.1692  DOG>
   5     405      0.1240      0.2812     -0.1572  DOG<
   6    2361      0.5739      0.4577      0.1162  DOG>
   7    1550      0.1084     -0.0064      0.1147  DOG>
   8    1237      0.0529     -0.0574      0.1104  DOG>
   9    2827      0.7560      0.6533      0.1027  DOG>
  10    1685      0.2288      0.1342      0.0947  DOG>
  11    1859      0.1582      0.0650      0.0932  DOG>
  12    1123      0.2402      0.1473      0.0929  DOG>
  13    2540      0.2696      0.1774      0.0922  DOG>
  14    1032      0.087

## Step 4 (Part B) — Explainer LLM: Llama 3 8B (local via Ollama)

The explainer LLM analyzes the neuron activation patterns to determine whether the top-ranked neurons are plausibly associated with the concept `dog`.*

We build a structured prompt containing:
- The top-15 neuron activation table from the previous step
- All 10 prompts labeled by class

The local **Llama 3 8B** model (served by Ollama, which uses MPS automatically on Apple
Silicon) is asked to interpret four questions:
1. Are these neurons tracking `dog` specifically, or something more general (animals, pets, animate beings)?
2. What do neurons with **positive** differences (DOG>) vs. **negative** differences (DOG<) suggest about what BERT encodes?
3. What **alternative hypotheses** could explain these differences given structurally matched prompts?
4. What **follow-up experiment** would disambiguate dog-specific from animal-general neurons?

In [ ]:
# Build the neuron table string
rows = []
for rank, idx in enumerate(top_indices, 1):
    i = idx.item()
    d = diff[i].item()
    direction = 'DOG>' if d > 0 else 'DOG<'
    rows.append(
        f'  {rank:>2}. Neuron {i:>4}: '
        f'dog={dog_mean[i].item():>8.4f}  '
        f'ctrl={control_mean[i].item():>8.4f}  '
        f'diff={d:>+9.4f}  ({direction})'
    )
table_str = '\n'.join(rows)

dog_list  = '\n'.join(f'  [DOG]     {p}' for p in dog_prompts)
ctrl_list = '\n'.join(f'  [CONTROL] {p}' for p in control_prompts)

prompt_lines = [
    'You are a mechanistic interpretability researcher analyzing BERT internal representations.',
    '',
    'Experiment: bert-base-uncased was run on 10 prompts (5 containing the word dog,',
    '5 structurally matched controls). Post-GELU activations from layer 8 BertIntermediate',
    '(3072 neurons) were captured and mean-pooled over token positions per prompt.',
    '',
    f'Top {TOP_N} most differentially active neurons (ranked by |dog_mean - control_mean|):',
    table_str,
    '',
    'Dog prompts:',
    dog_list,
    '',
    'Control prompts:',
    ctrl_list,
    '',
    'Questions:', #Here we plan to implement Lily Weng's prompt "summary" structure.
    '1. Are these neurons tracking dog specifically, or something more general (animals, pets,',
    '   animate things)? Support your answer with the activation magnitudes and directions.',
    '2. What do positive-diff neurons (DOG> - more active for dog prompts) vs negative-diff',
    '   neurons (DOG< - suppressed for dog prompts) suggest about what BERT is encoding?',
    '3. What alternative hypotheses could explain these differences given matched sentence frames?',
    '4. What follow-up experiment would disambiguate dog-specific from animal-general neurons?',
    '',
    'Reason carefully about what the specific neuron indices and activation magnitudes suggest.',
]
explainer_prompt = '\n'.join(prompt_lines)

print('Querying local Llama 3 8B via Ollama...')
print('-' * 60)
response = ollama.chat(
    model='llama3',
    messages=[{'role': 'user', 'content': explainer_prompt}]
)
explanation = response['message']['content']
print(explanation)

Querying local Llama 3 8B via Ollama...
------------------------------------------------------------
I'd be happy to help you analyze the results of this experiment.

**Question 1: Are these neurons tracking dog specifically, or something more general?**

After examining the activation magnitudes and directions, I would argue that these neurons are tracking something more general than just "dog". Here's why:

* Neurons 1, 2, and 4 (DOG>) have relatively high activation magnitudes (>0.2) and are consistently more active for dog prompts. This suggests that these neurons are sensitive to the presence of dogs in the input text.
* Neurons 3, 5, and 15 (DOG<) have lower activation magnitudes (<0.2) and are suppressed for dog prompts. This could indicate that these neurons are sensitive to other semantic features that are more common in control prompts (e.g., cats, rabbits, birds, horses, foxes).
* Neurons 6, 9, 11, 12, and 14 (DOG>) have moderate to high activation magnitudes (0.05-0.7) and 

## Step 5 — Results & Conclusion

Report the results and include concrete evidence that the experiment was actually carried out.*

The cell below collects all experiment outputs into a single structured report:

- **Section 1** — All 10 prompts with class labels (concrete evidence of what was run)
- **Section 2** — Class-level neuron activation statistics (mean, std, max, max-diff neuron)
- **Section 3** — The top-15 differentially active neuron table with real float values
- **Section 4** — The full Llama 3 8B explanation (evidence the explainer was invoked)
- **Section 5** — Experiment metadata (device, model, layer, timestamp)

The presence of specific 4-decimal neuron activation values and the Llama response text
together constitute the end-to-end evidence of execution.

In [9]:
SEP = '=' * 70

print(SEP)
print('BERT MECHANISTIC INTERPRETABILITY EXPERIMENT — FINAL REPORT')
print(f'Subject    : dog  |  Model: {MODEL_ID}  |  Layer: {LAYER} (BertIntermediate)')
print(f'Explainer  : Llama 3 8B (local via Ollama)')
print(f'Device     : {device}  |  Run: {ts}')
print(SEP)

print()
print('--- SECTION 1: PROMPTS USED ---')
for p in dog_prompts:
    print(f'  [DOG]     {p}')
for p in control_prompts:
    print(f'  [CONTROL] {p}')

print()
print('--- SECTION 2: CLASS ACTIVATION STATISTICS ---')
print(f'  Dog class    - mean: {dog_mean.mean().item():.4f}  std: {dog_mean.std().item():.4f}  max: {dog_mean.max().item():.4f}')
print(f'  Control class- mean: {control_mean.mean().item():.4f}  std: {control_mean.std().item():.4f}  max: {control_mean.max().item():.4f}')
print(f'  Max |diff|   : {diff.abs().max().item():.4f}  at neuron {diff.abs().argmax().item()}')

print()
print(f'--- SECTION 3: TOP {TOP_N} DIFFERENTIALLY ACTIVE NEURONS ---')
print('  Rank  Neuron    Dog Mean   Ctrl Mean        Diff  Dir')
print('  ' + '-' * 54)
for rank, idx in enumerate(top_indices, 1):
    i = idx.item()
    d = diff[i].item()
    direction = 'DOG>' if d > 0 else 'DOG<'
    print(f'  {rank:>4}  {i:>6}  {dog_mean[i].item():>10.4f}  {control_mean[i].item():>10.4f}  {d:>10.4f}  {direction}')

print()
print('--- SECTION 4: LLM EXPLAINER RESPONSE (Llama 3 8B via Ollama) ---')
for line in explanation.split('\n'):
    print(f'  {line}')

print()
print('--- SECTION 5: EXPERIMENT METADATA ---')
print(f'  BERT model      : {MODEL_ID}')
print(f'  Hook target     : encoder.layer[{LAYER}].intermediate (BertIntermediate, post-GELU)')
print(f'  Activation shape: (3072,) per prompt (mean-pooled over sequence tokens)')
print(f'  Dog prompts     : {len(dog_prompts)}')
print(f'  Control prompts : {len(control_prompts)}')
print(f'  Top-N neurons   : {TOP_N}')
print(f'  Explainer model : llama3 (Ollama local)')
print(f'  Device (BERT)   : {device}')
print(SEP)
print('END OF REPORT')
print(SEP)

BERT MECHANISTIC INTERPRETABILITY EXPERIMENT — FINAL REPORT
Subject    : dog  |  Model: bert-base-uncased  |  Layer: 8 (BertIntermediate)
Explainer  : Llama 3 8B (local via Ollama)
Device     : mps  |  Run: 2026-03-07 17:26:18

--- SECTION 1: PROMPTS USED ---
  [DOG]     The dog ran across the yard and barked loudly.
  [DOG]     She fed her dog a treat after the long walk.
  [DOG]     A large dog sat beside the front door waiting.
  [DOG]     He trained his dog to sit and stay on command.
  [DOG]     The dog curled up on the couch next to its owner.
  [CONTROL] The cat ran across the yard and meowed loudly.
  [CONTROL] She fed her rabbit a treat after the long walk.
  [CONTROL] A large bird sat beside the front door waiting.
  [CONTROL] He trained his horse to sit and stay on command.
  [CONTROL] The fox curled up on the couch next to its owner.

--- SECTION 2: CLASS ACTIVATION STATISTICS ---
  Dog class    - mean: -0.0410  std: 0.0577  max: 0.8553
  Control class- mean: -0.0420  std: 